# Visualización de outliers y distribuciones
El objetivo de este notebook es visualizar los outliers y distribuciones de todas las variables que vamos a utilizar.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [ ]:
base_dir = Path().resolve().parents[1]
data_dir = base_dir / 'data/processed'
df = pd.read_parquet(Path(data_dir / 'popularidad.parquet'))
pd.options.display.max_columns = None
df.columns

Tenemos muchas columnas con diferentes tipos de datos. Las numéricas y temporales serán fáciles de visualizar, mientras que las categóricas y textuales no tanto. Vamos ir una por una analizando cada variable. 

## id 
No merece la pena visualizar el id al ser un número asignado con el único propósito de identificar los juegos, por esto no se considera ni que tenga una distribución ni valores atípicos.

## name
Los nombres no tienen una distribución como tal, lo máximo que podemos hacer es visualizar la longitud de los mismos. Se observa que la media está en torno a los 13 caracteres, teniendo algunos valores atípicos con más de 100 caracteres.

In [ ]:
df_names = df.copy()
df_names["len_name"] = df_names["name"].apply(len)
df_names = df_names[df_names["len_name"] < 150]
fig = px.histogram(df_names, x = df_names["len_name"], hover_name= df_names["name"], color_discrete_sequence  = ["lightseagreen"], title="<b>Distribución de longitudes de nombres</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()

## short_description
No merece la pena visualizar la descripción de los juegos ya que no hay ninguna manera directa de ver distribuciones o valores atípicos.

## price_overview y price_range
La distribución de precios ya se muestra en el notebook dedicado a ello, además de experimentar con diferentes condiciones. Pero repasamos la distribución total de la variable. Destaca el rango de precios de 0.01€ a 4.99€ con más del 40% de los juegos. Además de que menos del 25% de los juegos cuesta más de 10€.

In [ ]:
fig = go.Figure()
orden = [
    "Free", "[0.01,4.99]",
    "[5.00,9.99]", "[10.00,14.99]",
    "[15.00,19.99]", "[20.00,29.99]",
    "[30.00,39.99]", ">40"]

values_df = df["price_range"].value_counts()
values_df = values_df.reindex(orden)

fig.add_trace(go.Pie(values= values_df.values, labels= values_df.index, sort = False))
fig.update_layout(
    width = 800,
    title = {
        "text": "Proporción de los rangos de precio",
        "x": 0.5, "y":0.9, "font":dict(family="Verdana", size = 20, weight="bold")
    })

fig.show()

## supported_languages
Esta es una variable que almacena un array de los idiomas que soporta un juego, podemos graficar el número de reseñas soportadas por cada juego:

In [ ]:
df_lang = df.explode('supported_languages')
df_counts = df_lang['supported_languages'].value_counts().reset_index()
df_counts.columns = ['idioma', 'cantidad']

print(f"hay {df_counts.shape[0]} idiomas, de los cuales {df_counts[df_counts["cantidad"] > 1000].shape[0]} son soportados por al menos el 95% de los juegos")

In [ ]:
df_counts = df_counts[df_counts["cantidad"]> 1000]
df_counts = df_counts.sort_values("cantidad", ascending= True)

fig = px.bar(df_counts, 
             y='idioma', 
             x='cantidad',
             title='Distribución de Idiomas Soportados',
             labels={'idioma': 'Idioma', 'cantidad': 'Número de juegos'})

fig.update_layout(
    width = 900,
    height = 600,
    title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)}
    )

fig.show()

Destaca el inglés, estando soportado por casi la totalidad de los juegos.

## developers y publishers
Vamos a intentar hacer lo mismo que con los idomas, pero hay muchos más desarrolladores y publishers que idomas, por lo que visualizaremos solo los que tengan más de cierto número de juegos.

In [ ]:
df_dev_counts = df.explode('developers')['developers'].value_counts().reset_index()
df_dev_counts.columns = ['developer', 'cantidad']

df_pub_counts = df.explode('publishers')['publishers'].value_counts().reset_index()
df_pub_counts.columns = ['publisher', 'cantidad']

porcentaje = round((df_dev_counts[df_dev_counts["cantidad"] > 1].shape[0] / df_dev_counts.shape[0])*100,2)
print(f"hay {df_dev_counts.shape[0]} developers, de los cuales {porcentaje}% tienen más de un juego")

porcentaje = round((df_pub_counts[df_pub_counts["cantidad"] > 1].shape[0] / df_pub_counts.shape[0])*100,2)
print(f"hay {df_pub_counts.shape[0]} publishers, de los cuales {porcentaje}% tienen más de un juego")

Al haber tantos developers y publishers visualizarlos sería una tarea muy complicada. Para hacernos una idea veremos tan solo los que tienen más juegos publicados, pero a la hora de construir nuestro modelo tendremos que pensar como manejarlo, ya que el manejo estandar de variables categóricas (una columna por valor) puede resultar en un modelo con demasiadas variables. 

In [ ]:
df_dev_counts = df_dev_counts[df_dev_counts["cantidad"] > 11].sort_values("cantidad", ascending=True)
df_pub_counts = df_pub_counts[df_pub_counts["cantidad"] > 17].sort_values("cantidad", ascending=True)
max_eje_x = max(df_dev_counts['cantidad'].max(), df_pub_counts['cantidad'].max())

fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=("Distribución de Developers", "Distribución de Publishers"),
                    horizontal_spacing=0.2)

fig.add_trace(
    go.Bar(y=df_dev_counts['developer'], x=df_dev_counts['cantidad'], 
           orientation='h', marker_color='gold', name="Devs"),
    row=1, col=1
)

fig.add_trace(
    go.Bar(y=df_pub_counts['publisher'], x=df_pub_counts['cantidad'], 
           orientation='h', marker_color='seagreen', name="Pubs"),
    row=1, col=2
)

fig.update_layout(height=800, width=1200, showlegend=False)
fig.update_xaxes(title_text="Número de juegos", range=[0, max_eje_x])
fig.show()

## categories y genres
Al igual que los anteriores, vamos a estudiar juntos estas variables. También son categóricas, pero esperamos menos valores que en los developers.

In [ ]:
df_cat_counts = df.explode('categories')['categories'].value_counts().reset_index()
df_cat_counts.columns = ['categorie', 'cantidad']

df_gen_counts = df.explode('genres')['genres'].value_counts().reset_index()
df_gen_counts.columns = ['genre', 'cantidad']

porcentaje = round((df_cat_counts[df_cat_counts["cantidad"] > 100].shape[0] / df_cat_counts.shape[0])*100,2)
print(f"hay {df_cat_counts.shape[0]} categories, de los cuales {porcentaje}% aparecen en 100 juegos o más")

porcentaje = round((df_gen_counts[df_gen_counts["cantidad"] > 100].shape[0] / df_gen_counts.shape[0])*100,2)
print(f"hay {df_gen_counts.shape[0]} genres, de los cuales {porcentaje}% aparecen en 100 juegos o más")

Hay significativamente menos categorías y géneros que developers y publishers, por lo que sí podríamos utilizar estas variables categórias de manera normal. Ahora visualicemos las que aparecen más veces.

In [ ]:
df_cat_counts = df_cat_counts[df_cat_counts["cantidad"] > 200].sort_values("cantidad", ascending=True)
df_gen_counts = df_gen_counts[df_gen_counts["cantidad"] > 1].sort_values("cantidad", ascending=True)

max_eje_x = max(df_cat_counts['cantidad'].max(), df_gen_counts['cantidad'].max())

fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=("Distribución de categorías", "Distribución de géneros"),
                    horizontal_spacing=0.2)

fig.add_trace(
    go.Bar(y=df_cat_counts['categorie'], x=df_cat_counts['cantidad'], 
           orientation='h', marker_color='hotpink', name="Devs"),
    row=1, col=1
)

fig.add_trace(
    go.Bar(y=df_gen_counts['genre'], x=df_gen_counts['cantidad'], 
           orientation='h', marker_color='slateblue', name="Pubs"),
    row=1, col=2
)

fig.update_layout(height=800, width=1200, showlegend=False)
fig.update_xaxes(title_text="Número de juegos", range=[0, max_eje_x])
fig.show()

## release_date
Ver la distribución de esta variable es sencillo

In [ ]:
df['release_date'] = pd.to_datetime(df['release_date'])

fig = px.histogram(df, x="release_date", 
                   title="<b>Distribución de fechas de lanzamiento</b>",
                   labels={'release_date': 'Fecha', 'count': 'Número de juegos'},
                   color_discrete_sequence  = ["lightcoral"]
                   )

fig.update_layout(
    title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)}, bargap=0.1, plot_bgcolor = "WhiteSmoke")
fig.show()

## recomendaciones_positivas y recomendaciones_negativas
La distribución es una cola larga, teniendo casi todos los juegos muy pocas reseñas

In [ ]:
limit = df[["recomendaciones_positivas", "recomendaciones_negativas"]].quantile(0.98).max()
bins = np.linspace(0, limit, 25)
pos_h, _ = np.histogram(df["recomendaciones_positivas"], bins=bins)
neg_h, _ = np.histogram(df["recomendaciones_negativas"], bins=bins)
labels = [f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins)-1)]

pos_log = np.log1p(pos_h)
neg_log = np.log1p(neg_h)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=labels, 
    x=neg_log * -1, 
    name="Negativas", 
    orientation="h", 
    marker_color="crimson",
    hovertemplate="Rango: %{y}<br>Juegos: %{customdata}",
    customdata=neg_h
))

fig.add_trace(go.Bar(
    y=labels, 
    x=pos_log, 
    name="Positivas", 
    orientation="h", 
    marker_color="seagreen",
    hovertemplate="Rango: %{y}<br>Juegos: %{customdata}",
    customdata=pos_h
))

tick_vals_raw = [0, 10, 100, 1000, 10000]
tick_vals_log = np.log1p(tick_vals_raw)

all_tick_vals = np.concatenate([-tick_vals_log[::-1], tick_vals_log])
all_tick_text = [str(x) for x in tick_vals_raw[::-1]] + [str(x) for x in tick_vals_raw]

fig.update_layout(
    title={"text": "<b>Recomendaciones (Escala Logarítmica)</b>", "x": 0.5},
    barmode="overlay",
    xaxis=dict(
        tickvals=all_tick_vals,
        ticktext=all_tick_text,
        title="Número de juegos (Escala Log)"
    ),
    yaxis=dict(title="Rango de recomendaciones"),
    height=700
)

fig.show()

## brillo
El brillo medio es un valor que va entre 0 y 255 (valores de negro y blanco), la media está entre 50 y 100 de brillo

In [ ]:
fig = px.histogram(df, x = df["brillo"], hover_name= df["name"], color_discrete_sequence  = ["crimson"], title="<b>Distribución del brillo de las cápsulas</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()

## yt_score
El yt_score es una ponderación normalizada, por lo que está entre 0 y 1. Destaca que casi el 25% de los juegos tiene un score de 0, es decir, no tienen ningún video.

In [ ]:
fig = px.histogram(df, x = df["yt_score"], hover_name= df["name"], color_discrete_sequence  = ["crimson"], title="<b>Distribución yt_score</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()